# 04 — Common Example (all three methods)

One culture / parallel prompt pair → SAE features, JLens readout, activation-patching layer curve.

This is the Placement-1 “small common example” pattern.

In [ ]:
# --- Colab / local bootstrap (Drive + wheels + swappable model) ---
# Drive layout expected:
#   MyDrive/multilingual-mechinterp/
#     dist/*.whl
#     data/all200questions_persianMiddleEastCulture.json
#     configs/  notebooks/  results/
#
# Edit MODEL_NAME in notebooks/colab_setup.py (Qwen2.5 now; Gemma later),
# or override below after bootstrap.

from pathlib import Path
import runpy

def _resolve_setup_script() -> Path:
    here = Path.cwd()
    candidates = [
        here / "colab_setup.py",
        here / "notebooks" / "colab_setup.py",
        here.parent / "notebooks" / "colab_setup.py",
        Path("/content/drive/MyDrive/multilingual-mechinterp/notebooks/colab_setup.py"),
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(
        "colab_setup.py not found. Mount Drive with the project folder, "
        "or open the notebook from the repo."
    )

_setup = runpy.run_path(str(_resolve_setup_script()))
globals().update({k: _setup[k] for k in _setup["EXPORTS"]})

# Session overrides (uncomment as needed):
# MODEL_NAME = "google/gemma-2-2b"
# MODEL_TRUST_REMOTE_CODE = False
# USE_TINY_OFFLINE = True   # demos without downloading HF weights

import matplotlib.pyplot as plt
import torch

from multilingual_mechinterp.utils import ensure_dir, load_config

cfg_path = CONFIG_DIR / "qwen25.yaml"
cfg = load_config(cfg_path) if cfg_path.exists() else {}
if "model" in cfg and not USE_TINY_OFFLINE:
    # keep notebook MODEL_NAME as source of truth; cfg is fallback metadata
    pass

print("Ready.")
print(" ROOT =", ROOT)
print(" DATA =", DATA_DIR)
print(" DIST =", DIST_DIR)
print(" MODEL =", MODEL_NAME, "| tiny=", USE_TINY_OFFLINE)

from multilingual_mechinterp.data import culture_prompt_pairs, load_culture_questions
from multilingual_mechinterp.jlens import TinyDecoder, fit, run_jlens
from multilingual_mechinterp.metrics import evaluate_sae
from multilingual_mechinterp.patching import TinyCausalLM, recommend_layers, run_patching
from multilingual_mechinterp.sae import train_tied_sae

OUT = ensure_dir(RESULTS_DIR / "comparison")


## Load experiment model

Uses `MODEL_NAME` from `colab_setup.py` (default **Qwen2.5**). Set `USE_TINY_OFFLINE=True` for demos without HF downloads.


In [ ]:
# Real model (Qwen now; change MODEL_NAME for Gemma later) OR tiny offline
# model = load_experiment_model()
# For gated Gemma: export HF_TOKEN=... or pass token=...

# Default path in analysis cells below uses tiny models for speed.
# Swap in `model = load_experiment_model()` when you are ready for Qwen/Gemma.
print("To load HF weights:", f"load_experiment_model({MODEL_NAME!r})")
print("Culture JSON:", culture_json_path(), "exists=", culture_json_path().exists())


## 1. Load one EN ↔ FA culture item

In [ ]:
path = ROOT / "data" / "all200questions_persianMiddleEastCulture.json"
if path.exists():
    pairs = culture_prompt_pairs(
        load_culture_questions(path, limit=1),
        source_lang="english",
        target_lang="persian",
        limit=1,
    )
    pair = pairs[0]
else:
    pair = {
        "source_prompt": "The capital of France is",
        "target_prompt": "پایتخت فرانسه",
        "source_answer": "Paris",
        "target_answer": "پاریس",
        "id": "demo",
    }

src = pair["source_prompt"][:220]
tgt = pair["target_prompt"][:220]
answer = pair["source_answer"]
print("id=", pair.get("id"))
print("EN answer:", answer)
print("FA answer:", pair.get("target_answer"))
print("--- source ---\n", src[:400])
print("--- target ---\n", tgt[:400])

## 2. SAE on shared residual-like features (offline)

In [ ]:
torch.manual_seed(0)
# Stand-in activations (replace with extract_residual_activations on a real LM)
acts = torch.randn(2000, 64)
sae_result = train_tied_sae(acts, ratio=2, alpha=8.6e-4, n_epochs=2, batch_size=256)
sae_metrics = evaluate_sae(sae_result.sae, acts[:1000])
print("SAE", sae_metrics)

with torch.no_grad():
    codes = sae_result.sae.encode(acts[:256])
topk = torch.topk(codes.mean(0), k=10)

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar([str(int(i)) for i in topk.indices], topk.values.numpy(), color="#F58518")
ax.set_title("SAE top features (mean act)")
ax.set_xlabel("feature id")
plt.tight_layout()
plt.show()

## 3. JLens on the English prompt

In [ ]:
jlens_model = TinyDecoder(n_layers=4, d_model=16, seed=1)
lens = fit(
    jlens_model,
    [src[:80], "Culture and tradition in Iran include Nowruz and Hafez"],
    source_layers=[0, 1, 2],
    target_layer=3,
    skip_first=0,
    dim_batch=8,
)
jlens_out = run_jlens(
    jlens_model, src[:80], lens=lens, layers=[0, 1, 2],
    positions=[-1], top_k=5, use_jacobian=True,
)

fig, ax = plt.subplots(figsize=(6, 3.5))
for layer, toks in jlens_out.top_tokens.items():
    ax.plot([t["logit"] for t in toks], marker="o", label=f"L{layer}")
ax.set_title("JLens top logits by layer")
ax.set_xlabel("rank")
ax.set_ylabel("logit")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
jlens_out.top_tokens

## 4. Activation patching EN → FA (choose layers)

In [ ]:
patch_model = TinyCausalLM(n_layers=6, d_model=32, seed=2)
patch = run_patching(
    patch_model,
    source_prompt=src[:120],
    target_prompt=tgt[:120],
    answer=str(answer).split()[0],
)
swap_layers = recommend_layers(patch, top_k=3, min_effect=-1.0)

fig, ax = plt.subplots(figsize=(6, 3.5))
xs = sorted(patch.scores)
ax.plot(xs, [patch.scores[L] for L in xs], marker="o")
ax.axhline(0, color="k", lw=0.7, alpha=0.4)
for L in swap_layers:
    ax.axvline(L, color="#E45756", ls=":", alpha=0.8)
ax.set_title(f"Patching ΔP — recommend {swap_layers}")
ax.set_xlabel("start layer")
ax.set_ylabel("ΔP(answer)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
fig.savefig(OUT / "common_example_patching.png", dpi=150)
plt.show()

summary = {
    "pair_id": pair.get("id"),
    "sae_fvu": sae_metrics["fvu"],
    "sae_l0": sae_metrics["mean_l0"],
    "jlens_layers": jlens_out.layers,
    "patch_best_layer": patch.best_layer,
    "layers_to_swap": swap_layers,
}
summary